In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch

VIDEO = sorted(Path("sample-500/files").glob("*.avi"))[0]
N_FRAMES = 4
SEED = 42
SCALE = 0.25
THRESHOLD = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [ ]:
BASE = "https://raw.githubusercontent.com/lseventeen/FR-UNet/1856bc206055fdfccde184d0ebf9ac8c3207387d"
CACHE = Path(".cache/fr_unet")
CACHE.mkdir(parents=True, exist_ok=True)
for source, name in [("models/fr_unet.py", "fr_unet.py"), ("models/utils.py", "utils.py"),
                     ("LICENSE", "LICENSE"), ("pretrained_weights/DRIVE/checkpoint-epoch40.pth", "drive.pth")]:
    if not (CACHE / name).exists():
        urlretrieve(f"{BASE}/{source}", CACHE / name)

utils = CACHE / "utils.py"
utils.write_text(utils.read_text().replace("from timm.models.layers import trunc_normal_",
                                          "from torch.nn.init import trunc_normal_"))
(CACHE / "__init__.py").touch()
sys.path.insert(0, str(CACHE.parent.resolve()))
from fr_unet.fr_unet import FR_UNet

with torch.serialization.safe_globals([(dict, "bunch.Bunch")]):
    checkpoint = torch.load(CACHE / "drive.pth", map_location="cpu", weights_only=True)
model = FR_UNet().to(DEVICE).eval()
model.load_state_dict({k.removeprefix("module."): v for k, v in checkpoint["state_dict"].items()});

In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
indices = np.sort(np.random.default_rng(SEED).choice(count, N_FRAMES, replace=False))
frames = []
for index in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
    _, frame = cap.read()
    frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()

In [ ]:
probabilities = []
with torch.inference_mode():
    for frame in frames:
        height, width = frame.shape[:2]
        image = Image.fromarray(frame).resize((round(width * SCALE), round(height * SCALE)), Image.Resampling.LANCZOS)
        gray = np.asarray(image.convert("L"), dtype=np.float32)
        gray = (gray - gray.min()) / max(float(np.ptp(gray)), 1.0)
        h, w = gray.shape
        padded = np.pad(gray, ((0, -h % 8), (0, -w % 8)), mode="edge")
        x = torch.from_numpy(padded)[None, None].to(DEVICE)
        p = model(x).sigmoid()[0, 0, :h, :w].cpu().numpy()
        probabilities.append(cv2.resize(p, (width, height), interpolation=cv2.INTER_LINEAR))

In [ ]:
fig, axes = plt.subplots(len(frames), 3, figsize=(12, 3 * len(frames)), squeeze=False)
for frame, p, row in zip(frames, probabilities, axes):
    mask = p >= THRESHOLD
    overlay = frame.copy()
    overlay[mask] = (0.6 * overlay[mask] + 0.4 * np.array([0, 255, 100])).astype(np.uint8)
    for ax, image in zip(row, (frame, p, overlay)):
        ax.imshow(image, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
plt.tight_layout()
plt.show()